# Titanic survival prediction

A cleaned end-to-end notebook for feature engineering, ticket-grouped CatBoost cross-validation, and Kaggle submission generation. It uses the reusable code in `titanic_survival/`.


## 1. Load Kaggle data

Download `train.csv` and `test.csv` from the [competition data page](https://www.kaggle.com/competitions/titanic/data) and place both inside `data/raw/`. If running on Kaggle, the notebook can also read from `/kaggle/input/competitions/titanic/`.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'titanic_survival').exists():
    ROOT = ROOT.parent
if not (ROOT / 'titanic_survival').exists():
    raise RuntimeError('Run this notebook from the cloned project root or notebooks/ directory.')
sys.path.insert(0, str(ROOT))

from titanic_survival.pipeline import (
    DEFAULT_GRID, DEFAULT_PARAMS, FEATURES, engineer_features,
    run_group_cv, create_submission,
)

kaggle_dir = Path('/kaggle/input/competitions/titanic')
DATA = Path(os.environ.get('TITANIC_DATA_DIR', kaggle_dir if kaggle_dir.exists() else ROOT / 'data' / 'raw'))
if not (DATA / 'train.csv').is_file() or not (DATA / 'test.csv').is_file():
    raise FileNotFoundError(f'Place Kaggle train.csv and test.csv in {DATA}')
train = pd.read_csv(DATA / 'train.csv')
test = pd.read_csv(DATA / 'test.csv')
print('Train:', train.shape, 'Test:', test.shape)

## 2. Inspect the data

Check missing values and the label distribution before modeling.

In [ ]:
display(pd.DataFrame({
    'missing_train': train.isna().sum(),
    'missing_test': test.isna().sum()
}).fillna(0).astype(int))
print('Survival rate:', round(train['Survived'].mean(), 4))

## 3. Feature engineering

Extract passenger titles and family size, then use training-fold statistics for missing age and fare values. The final model uses the eight columns defined by `FEATURES`.


In [ ]:
example = engineer_features(train)
display(example[['Name','Title','SibSp','Parch','FamilySize','IsAlone']].head(10))
print('Maintained model columns:', FEATURES)

## 4. Recorded CatBoost experiments

The table below contains saved results from eight previous CatBoost configurations. They are five-fold `GroupKFold` validation accuracies, **not** an independently reproduced benchmark or Kaggle leaderboard score. The cleaned pipeline may produce different values when rerun with the competition CSV files.


In [ ]:
historical = pd.read_csv(ROOT / 'results' / 'recorded_cv_results.csv')
display(historical)

## 5. Ticket-grouped cross-validation

Set `RUN_FULL_CV = True` to evaluate eight configurations using five folds. The default runs one small model with three folds as a quick configuration check.


In [ ]:
RUN_FULL_CV = False
small_grid = {'iterations': [30], 'depth': [5], 'learning_rate': [0.02], 'loss_function': ['Logloss']}
results = run_group_cv(train,
    param_grid=DEFAULT_GRID if RUN_FULL_CV else small_grid,
    n_splits=5 if RUN_FULL_CV else 3)
display(results)

## 6. Train the final model and generate a Kaggle submission

Fit CatBoost using the selected configuration and the same eight model features used during cross-validation, then save `PassengerId,Survived` predictions.


In [ ]:
submission = create_submission(train, test, parameters=DEFAULT_PARAMS)
output = ROOT / 'outputs' / 'submission.csv'
output.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(output, index=False)
print('Saved', output, '| rows:', len(submission))
display(submission.head())

## 7. Interpret the results

The best recorded run had **0.8248 mean cross-validation accuracy** and **0.0460 standard deviation** across folds. These values are saved experimental results, not measured leaderboard performance. Rerun cross-validation using the downloaded competition data to obtain results for the current code.
